#  Süt Verimi Tahmini — Projenin İlk Gerçek ML Modeli

**Proje:** AI Destekli Cepte Veterinerlik


**Amaç:** Süt ineklerinin özelliklerine (ırk, kaçıncı doğum, mevsim vb.) bakarak
süt verimini (MY - Milk Yield, kg) tahmin eden bir regresyon modeli kurmak.

**Veri:** Mendeley Data'dan alınan, Çek Cumhuriyeti'ndeki süt ineklerine ait
gerçek bir akademik veri seti (723 satır, 22 sütun). Bu, antrenman turundan
farklı olarak projenin gerçek bir parçasıdır.

In [1]:
from google.colab import files
yuklenen = files.upload()

Saving Dataset_cow_performance.xlsx to Dataset_cow_performance.xlsx


In [2]:
import pandas as pd

df = pd.read_excel("Dataset_cow_performance.xlsx", sheet_name="Dataset")

print("Satır ve sütun sayısı:", df.shape)
df.head()

Satır ve sütun sayısı: (723, 22)


,ID,ID Cow,ID cow with Codex,Calving Date,ID Breed,Breed,Calving Season,Year of Lactation,Parity,Method of Performance Control Database,...,CalvInt,SP,GZW/SIH,MY (kg),F (kg),P (kg),L (kg),F (%),P (%),L (%)
0,4100266939,192779,192779 941,2018-10-12,1,C100,4,18,1,AA,...,459,174,96.0,5800,249,233,309,4.33,4.03,5.15
1,4100211911,161742,161742 941,2018-10-14,1,C100,4,18,5,A4,...,409,124,NaN,5936,203,181,295,3.41,3.06,4.91
2,4100266939,192717,192717 941,2018-08-03,1,C100,3,18,1,AA,...,380,95,93.0,5760,232,216,298,4.08,3.74,4.96
3,4100260323,199537,199537 941,2018-06-13,1,C100,3,18,1,AA,...,386,101,93.0,5595,237,200,300,4.26,3.61,4.99
4,4100260323,199527,199527 941,2018-06-13,1,C100,3,18,1,AA,...,379,94,110.0,5894,247,204,308,4.22,3.49,5.11


veriyi Excel'den, Dataset sayfasından okuduk, 723 satır 22 sütun, ilk satırlara baktık

In [3]:
print("Sütunlar:")
print(df.columns.tolist())
print()

df.info()

Sütunlar:
['ID', 'ID Cow', 'ID cow with Codex', 'Calving Date', 'ID Breed', 'Breed', 'Calving Season', 'Year of Lactation', 'Parity', 'Method of Performance Control Database', 'Farm ID', 'Stable ID', 'CalvInt', 'SP', 'GZW/SIH', 'MY (kg)', 'F (kg)', 'P (kg)', 'L (kg)', 'F (%)', 'P (%)', 'L (%)']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 723 entries, 0 to 722
Data columns (total 22 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   ID                                      723 non-null    int64         
 1   ID Cow                                  723 non-null    int64         
 2   ID cow with Codex                       723 non-null    object        
 3   Calving Date                            723 non-null    datetime64[ns]
 4   ID Breed                                723 non-null    int64         
 5   Breed                                   723 non-null    objec

farkettiysek 723 satırın hepsi non-null(yani hepsi dolu) ancak 14.sütuna geldiğimizde 612 dolu değer oldugunu söylüyor gerikalan 111 değer boş kalmış veri dünyasında böyle durumlarla karsılasınca ya bu 111 değeri atarız ya da doldururuz.

22 sütunu da parçalamamız gerekicek çünkü model eğitmek istediğimiz için bazı sütunları atıcaz ki modelimiz kopya cekmiş olmasın, birebir alakalı olanları zaten cıkarıcaz(süt bileşenlerini)

In [4]:
df.describe()

,ID,ID Cow,Calving Date,ID Breed,Calving Season,Year of Lactation,Parity,Farm ID,Stable ID,CalvInt,SP,GZW/SIH,MY (kg),F (kg),P (kg),L (kg),F (%),P (%),L (%)
count,7.230000e+02,723.000000,723,723.000000,723.000000,723.0,723.000000,723.000000,7.230000e+02,723.000000,723.000000,612.000000,723.000000,723.000000,723.000000,723.000000,723.000000,723.000000,723.000000
mean,4.101570e+09,174485.344398,2018-06-21 05:46:33.360995840,1.582296,2.409405,18.0,2.734440,4.988935,4.052574e+09,422.970954,141.857538,96.955719,7627.842324,301.113416,268.369295,389.529737,3.966750,3.534232,4.947289
min,4.100006e+09,128684.000000,2018-01-01 00:00:00,1.000000,1.000000,18.0,1.000000,1.000000,0.000000e+00,0.000000,0.000000,75.000000,5572.000000,179.000000,169.000000,275.000000,2.930000,2.860000,4.280000
25%,4.100062e+09,163196.000000,2018-03-19 00:00:00,1.000000,1.000000,18.0,2.000000,3.000000,4.100052e+09,378.500000,94.000000,91.000000,6664.000000,262.000000,235.000000,339.000000,3.760000,3.400000,4.860000
50%,4.100260e+09,174896.000000,2018-06-13 00:00:00,1.000000,2.000000,18.0,2.000000,5.000000,4.100260e+09,408.000000,123.000000,96.000000,7445.000000,294.000000,263.000000,380.000000,3.970000,3.520000,4.960000
75%,4.100267e+09,185247.000000,2018-09-14 00:00:00,2.000000,3.000000,18.0,4.000000,8.000000,4.100267e+09,456.500000,171.500000,103.000000,8302.500000,332.000000,292.000000,424.500000,4.170000,3.670000,5.050000
max,5.100012e+09,203536.000000,2018-12-31 00:00:00,5.000000,4.000000,18.0,6.000000,9.000000,5.100012e+09,1134.000000,746.000000,120.000000,12504.000000,566.000000,443.000000,630.000000,5.850000,4.580000,5.380000
std,3.718409e+07,15045.988180,NaN,0.867905,1.110773,0.0,1.424466,2.648867,3.001255e+08,99.461698,75.565645,7.866955,1260.518594,52.188221,45.670663,65.682778,0.340959,0.222042,0.154869


In [5]:
# Gereksiz ve kopya-riskli sütunları at
atilacaklar = ["ID", "ID Cow", "ID cow with Codex", "Calving Date",
               "Farm ID", "Stable ID", "Method of Performance Control Database",
               "GZW/SIH", "F (kg)", "P (kg)", "L (kg)", "F (%)", "P (%)", "L (%)"]

df_temiz = df.drop(columns=atilacaklar)

print("Kalan sütunlar:", df_temiz.columns.tolist())
print("Yeni boyut:", df_temiz.shape)

Kalan sütunlar: ['ID Breed', 'Breed', 'Calving Season', 'Year of Lactation', 'Parity', 'CalvInt', 'SP', 'MY (kg)']
Yeni boyut: (723, 8)


In [7]:
# ID Breed'i de at (unutmuşuz), sonra One-Hot yap
df_temiz2 = df_temiz.drop(columns=["ID Breed"])

df_hazir = pd.get_dummies(df_temiz2, columns=["Breed", "Calving Season"], drop_first=True)

X = df_hazir.drop("MY (kg)", axis=1)
y = df_hazir["MY (kg)"]

print("X boyutu:", X.shape)
print("Sütunlar:", X.columns.tolist())

X boyutu: (723, 98)
Sütunlar: ['Year of Lactation', 'Parity', 'CalvInt', 'SP', 'Breed_C40RX', 'Breed_C50R', 'Breed_C50X', 'Breed_C53R', 'Breed_C58R', 'Breed_C58X', 'Breed_C58XA', 'Breed_C60R', 'Breed_C61XA', 'Breed_C61XH', 'Breed_C63RX', 'Breed_C67H', 'Breed_C67R', 'Breed_C68R', 'Breed_C69H', 'Breed_C69R', 'Breed_C69X', 'Breed_C70R', 'Breed_C70X', 'Breed_C71R', 'Breed_C71X', 'Breed_C72R', 'Breed_C72X', 'Breed_C75A', 'Breed_C75H', 'Breed_C75R', 'Breed_C75X', 'Breed_C76R', 'Breed_C77R', 'Breed_C77X', 'Breed_C78A', 'Breed_C78H', 'Breed_C78R', 'Breed_C79A', 'Breed_C79H', 'Breed_C79R', 'Breed_C79X', 'Breed_C80A', 'Breed_C80R', 'Breed_C80X', 'Breed_C81H', 'Breed_C81R', 'Breed_C81X', 'Breed_C82A', 'Breed_C82H', 'Breed_C83A', 'Breed_C83R', 'Breed_C83X', 'Breed_C84H', 'Breed_C84R', 'Breed_C84X', 'Breed_C85A', 'Breed_C85R', 'Breed_C85X', 'Breed_C86A', 'Breed_C86H', 'Breed_C86R', 'Breed_C86X', 'Breed_C87A', 'Breed_C87H', 'Breed_C87R', 'Breed_C88A', 'Breed_C88H', 'Breed_C88R', 'Breed_C88X', 'Breed

In [8]:
print("Breed'deki farklı değer sayısı:", df_temiz["Breed"].nunique())
print(df_temiz["Breed"].value_counts())

Breed'deki farklı değer sayısı: 92
Breed
C100     415
R100      33
C85R      18
C50R      18
C88R      10
        ... 
X60C       1
X62C       1
X63C       1
Y04CA      1
Y07C       1
Name: count, Length: 92, dtype: int64


In [9]:
# En sık 4 ırkı tut, gerisini "Diğer" yap
en_sik = df_temiz["Breed"].value_counts().nlargest(4).index
df_temiz["Breed_grup"] = df_temiz["Breed"].apply(lambda x: x if x in en_sik else "Diger")

# Artık eski Breed'i atıp yeni grubu kullanacağız
df_temiz2 = df_temiz.drop(columns=["ID Breed", "Breed"])

df_hazir = pd.get_dummies(df_temiz2, columns=["Breed_grup", "Calving Season"], drop_first=True)

X = df_hazir.drop("MY (kg)", axis=1)
y = df_hazir["MY (kg)"]

print("X boyutu:", X.shape)
print("Sütunlar:", X.columns.tolist())

X boyutu: (723, 11)
Sütunlar: ['Year of Lactation', 'Parity', 'CalvInt', 'SP', 'Breed_grup_C50R', 'Breed_grup_C85R', 'Breed_grup_Diger', 'Breed_grup_R100', 'Calving Season_2', 'Calving Season_3', 'Calving Season_4']


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Böl
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Kur ve eğit
model = LinearRegression()
model.fit(X_train, y_train)

# Değerlendir
tahminler = model.predict(X_test)
mae = mean_absolute_error(y_test, tahminler)
r2 = r2_score(y_test, tahminler)

print("MAE:", round(mae, 2), "kg")
print("R²:", round(r2, 3))

MAE: 899.14 kg
R²: 0.097


MAE: 899 kg — yani model ortalama 899 kg yanılıyor. Hatırla, süt verimi ortalaması 7627'ydi, yani ~%12 hata. Fena değil ama harika da değil.

R²: 0.097 — işte asıl konuşulacak sayı bu. R² 1'e yakın olsa iyi, 0'a yakınsa kötü demiştik. 0.097 çok düşük — yani model, verimdeki değişimin sadece ~%10'unu açıklayabiliyor, %90'ı hâlâ karanlıkta

yani ırkı cinsi ve diğer eklediğimiz özellikler tek basına hesaplamaya tahmin etmeye yetmiyor olumsuz gibi gözükğyor bu ama bi şeyleri öğrendiğim için de gayet iyi bi sonuç gibi de geliyor:)

In [11]:
from sklearn.ensemble import RandomForestRegressor

# Güçlü model kur ve eğit
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Değerlendir
rf_tahmin = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_tahmin)
rf_r2 = r2_score(y_test, rf_tahmin)

print("RandomForest MAE:", round(rf_mae, 2), "kg")
print("RandomForest R²:", round(rf_r2, 3))
print()
print("Karşılaştırma:")
print("Linear Regression → R²: 0.097")
print("Random Forest      → R²:", round(rf_r2, 3))


RandomForest MAE: 989.21 kg
RandomForest R²: -0.194

Karşılaştırma:
Linear Regression → R²: 0.097
Random Forest      → R²: -0.194


R² negatif ne demek? R² 0 demek "model, hiç düşünmeden hep ortalamayı söylemek kadar iyi." Negatif demek ise "model, ortalamayı söylemekten bile daha kötü tahmin yapıyor." Yani RandomForest bu veride resmen zarar veriyor — hiç model kurmasan, her ineğe "7627 kg" desen daha isabetli olurdu. Bu, modelin veriye "aşırı uyum" (overfitting) gösterip test verisinde çuvalladığının işareti.

# Linear ve RandomForest denedim → ikisi de düşük (0.097 ve -0.194) → sorun veride, çünkü güçlü model bile başaramadı → ırk/parity/mevsim yetmiyor, yem ve sağlık verisi lazım → daha kapsamlı veri gerekiyor.


##  Kapanış — Süt Verimi Tahmini Denemesi

Bu notebook'ta, gerçek bir akademik veri seti (Mendeley Data — Çek Cumhuriyeti
süt inekleri, 723 kayıt) üzerinde süt verimini (MY, kg) tahmin eden bir regresyon
modeli kurmayı denedim.

### İzlenen adımlar
1. Veriyi Excel'den okuma ve tanıma (`shape`, `info`, `describe`)
2. Gereksiz kimlik sütunlarını ve kopya riski taşıyan süt bileşenlerini (F/P/L) çıkarma
3. Eksik veri içeren sütunu (GZW/SIH) atma
4. 92 farklı ırk kodunu, en sık 4 ırk + "Diğer" olarak gruplama
5. Metin sütunlarını One-Hot Encoding ile sayıya çevirme
6. Veriyi bölme, iki farklı model eğitme ve karşılaştırma

### Sonuçlar
| Model | MAE | R² |
|---|---|---|
| Linear Regression | 899 kg | 0.097 |
| Random Forest | 989 kg | -0.194 |

### Temel Bulgu
Her iki modelin de düşük (hatta Random Forest'ın negatif) skor vermesi, sorunun
**algoritmada değil, veri setinin kapsamında** olduğunu gösterdi. Elimizdeki
"adil" özellikler (ırk, parity, mevsim, laktasyon yılı), süt verimini tahmin
etmek için tek başına yeterli bilgi taşımıyor. Verimi asıl belirleyen faktörler
(yem miktarı, hayvan sağlığı, bakım kalitesi) bu veri setinde bulunmuyor.

### Çıkardığım Ders
Bir modelin başarısız olması, çalışmanın başarısız olduğu anlamına gelmez. İki
farklı modeli karşılaştırarak, düşük performansın sebebini (model mi, veri mi?)
ayırt edebildim. Bu, veri biliminde dürüst bir değerlendirmenin nasıl yapıldığını
gösteren değerli bir deneyim oldu.

**Bir sonraki adım:** Verim tahmini için yem/sağlık gibi ek özellikler içeren daha
kapsamlı bir veri setine ihtiyaç var. Alternatif olarak, projenin diğer ML
özelliğine (anomali tespiti) geçilebilir.
